# M5.5 · RL temelleri: value iteration, REINFORCE

Pekiştirmeli öğrenmenin iki ucu: modeli bilinen bir MDP'de dinamik programlama, bilinmeyen bir ortamda policy gradient. İki durumlu bir MDP'de value iteration'ı elle yürütüyor, saf Python'da kaygan bir ızgara dünyası için value iteration ve policy evaluation yazıyor, NumPy'da Bellman operatörünü einsum'la ve politika değerini doğrusal sistem çözerek hesaplıyor, PyTorch'ta REINFORCE ile bir zincir ortamını öğreniyor ve en sonda 64 ortamı paralel yürütüp getirileri tek bir matris çarpımıyla hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-5-rl.html](https://m-gonul.github.io/pytorch-lab/m5-5-rl.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_5_rl.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef frozen_lake():\n    grid = [\"SFFF\", \"FHFH\", \"FFFH\", \"HFFG\"]\n    nS, nA = 16, 4\n    moves = [(0, -1), (1, 0), (0, 1), (-1, 0)]\n    P = {s: {a: [] for a in range(nA)} for s in range(nS)}\n    for s in range(nS):\n        r, c = divmod(s, 4)\n        for a in range(nA):\n            if grid[r][c] in \"HG\":\n                P[s][a] = [(1.0, s, 0.0, True)]\n                continue\n            for b in ((a - 1) % 4, a, (a + 1) % 4):\n                dr, dc = moves[b]\n                nr, nc = min(max(r + dr, 0), 3), min(max(c + dc, 0), 3)\n                s2 = nr * 4 + nc\n                P[s][a].append((1 / 3, s2, float(grid[nr][nc] == \"G\"), grid[nr][nc] in \"HG\"))\n    return P\n\n\ndef to_mats(P, nS, nA):\n    T = np.zeros((nS, nA, nS)); R = np.zeros((nS, nA)); D = np.zeros((nS, nA, nS))\n    for s in range(nS):\n        for a in range(nA):\n            for p, s2, r, d in P[s][a]:\n                R[s, a] += p * r\n                if not d:\n                    T[s, a, s2] += p\n    return T, R\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    P2 = {0: {0: [(1.0, 0, 0.0, False)], 1: [(1.0, 1, 1.0, False)]}, 1: {0: [(1.0, 1, 2.0, False)], 1: [(1.0, 0, 0.0, False)]}}\n\n    def tiny():\n        V, pi = need(ns, \"value_iteration\")(P2, 2, 2, 0.5)\n        close(V, [3, 4], atol=1e-8); ensure(list(pi) == [1, 0], f\"politika {pi}\")\n    R_.test(\"iki durumlu MDP (elle örnek)\", tiny)\n    P = frozen_lake()\n    T, R = to_mats(P, 16, 4)\n\n    def fl():\n        V, pi = need(ns, \"value_iteration\")(P, 16, 4, 0.99)\n        Vr = np.zeros(16)\n        for _ in range(5000):\n            Vr = (R + 0.99 * T @ Vr).max(1)\n        close(V, Vr, atol=1e-6)\n        Pp = T[np.arange(16), pi]; Rp = R[np.arange(16), pi]\n        Vpi = np.linalg.solve(np.eye(16) - 0.99 * Pp, Rp)\n        close(Vpi, Vr, atol=1e-6, what=\"açgözlü politikanın değeri = V*\")\n        return f\"V*(başlangıç) = {V[0]:.4f}: hedefe varma olasılığı (γ = 0.99 ile indirimli)\"\n    R_.test(\"kaygan buzlu göl: V* ve optimal politika\", fl)\n\n    def pe():\n        pol = [2] * 16\n        V = need(ns, \"policy_evaluation\")(P, pol, 16, 0.9)\n        Pp = T[np.arange(16), pol]; Rp = R[np.arange(16), pol]\n        close(V, np.linalg.solve(np.eye(16) - 0.9 * Pp, Rp), atol=1e-6)\n    R_.test(\"policy_evaluation (hep sağ)\", pe)\n\n    def qv():\n        V = [0.5] * 16\n        close(need(ns, \"q_values\")(P, V, 14, 4, 0.9), R[14] + 0.9 * T[14] @ np.array(V))\n    R_.test(\"q_values (hedefe komşu durum)\", qv)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"bellman_q döngüsüz\", lambda: max_loops(src, 0, func=\"bellman_q\"))\n    R_.test(\"policy_eval_exact döngüsüz\", lambda: max_loops(src, 0, func=\"policy_eval_exact\"))\n    R_.test(\"policy_iteration: tek döngü\", lambda: max_loops(src, 1, func=\"policy_iteration\"))\n    R_.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    S, A = 12, 3\n    P = rng.random((S, A, S)) ** 4; P /= P.sum(2, keepdims=True)\n    R = rng.normal(size=(S, A))\n    V = rng.normal(size=S)\n\n    def bq():\n        Q = np.zeros((S, A))\n        for s in range(S):\n            for a in range(A):\n                Q[s, a] = R[s, a] + 0.9 * sum(P[s, a, t] * V[t] for t in range(S))\n        close(need(ns, \"bellman_q\")(V, P, R, 0.9), Q)\n    R_.test(\"bellman_q\", bq)\n\n    def pe():\n        pi = rng.random((S, A)); pi /= pi.sum(1, keepdims=True)\n        Vp = need(ns, \"policy_eval_exact\")(P, R, pi, 0.9)\n        Q = R + 0.9 * P @ Vp\n        close(Vp, (pi * Q).sum(1), what=\"Bellman beklenti denklemi\")\n    R_.test(\"policy_eval_exact (stokastik π)\", pe)\n\n    def pi_():\n        V, pol = need(ns, \"policy_iteration\")(P, R, 0.95)\n        Vr = np.zeros(S)\n        for _ in range(3000):\n            Vr = (R + 0.95 * P @ Vr).max(1)\n        close(V, Vr, atol=1e-8, what=\"V* (value iteration ile)\")\n        ensure(np.array_equal(pol, (R + 0.95 * P @ Vr).argmax(1)), \"optimal politika\")\n        return f\"policy iteration {S} durumda yakınsadı; V*(0) = {V[0]:.4f}\"\n    R_.test(\"policy_iteration = value iteration\", pi_)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    Chain = need(ns, \"Chain\", \"sınıf\")\n    R.test(\"discounted_returns\", lambda: close(need(ns, \"discounted_returns\")([0.0, 0.0, 1.0], 0.5), [0.25, 0.5, 1.0]))\n\n    def ep():\n        torch.manual_seed(0)\n        pi = need(ns, \"Policy\", \"sınıf\")()\n        logps, rewards = need(ns, \"run_episode\")(pi, Chain())\n        ensure(logps.dim() == 1 and len(logps) == len(rewards) and logps.requires_grad, \"logps (T,) ve gradyanlı olmalı\")\n        ensure(all(r == 0 for r in rewards[:-1]), \"ödül yalnız son adımda olabilir\")\n    R.test(\"run_episode\", ep)\n\n    def train():\n        wins = 0\n        for seed in (0, 1, 2):\n            torch.manual_seed(seed)\n            pi = need(ns, \"Policy\", \"sınıf\")()\n            need(ns, \"train_reinforce\")(pi, Chain())\n            env = Chain(); obs = env.reset(); steps = 0\n            with torch.no_grad():\n                while True:\n                    obs, r, done = env.step(pi(obs).argmax(-1)); steps += 1\n                    if done.item():\n                        break\n            wins += int(env.s.item() == 7 and steps == 4)\n        ensure(wins >= 2, f\"3 tohumdan {wins} tanesinde açgözlü politika 4 adımda hedefe gitti\")\n        return f\"3 tohumdan {wins}'inde optimal politika (4 adımda sağ uç)\"\n    R.test(\"REINFORCE zinciri çözüyor\", train)\n",
 "e": "import time\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"discounted_returns_vec döngüsüz\", lambda: max_loops(src, 0, func=\"discounted_returns_vec\"))\n    R.test(\"rollout: tek döngü (zaman)\", lambda: max_loops(src, 1, func=\"rollout\"))\n    Chain, make_policy = need(ns, \"Chain\", \"sınıf\"), need(ns, \"make_policy\")\n\n    def dr():\n        g = torch.Generator().manual_seed(0)\n        r = (torch.rand(5, 12, generator=g, dtype=torch.float64) < 0.2).double()\n        want = torch.zeros_like(r)\n        for b in range(5):\n            G = 0.0\n            for t in reversed(range(12)):\n                G = r[b, t] + 0.9 * G\n                want[b, t] = G\n        close(need(ns, \"discounted_returns_vec\")(r, 0.9), want)\n    R.test(\"discounted_returns_vec = geriye döngü\", dr)\n\n    def ro():\n        torch.manual_seed(0)\n        env = Chain(B=16)\n        logp, r, mask = need(ns, \"rollout\")(make_policy(), env, 30)\n        ensure(tuple(logp.shape) == tuple(r.shape) == tuple(mask.shape) == (16, 30), \"şekiller (B, T)\")\n        ensure(torch.all(mask[:, 0] == 1) and torch.all(mask[:, 1:] <= mask[:, :-1]), \"mask başta 1, sonra artmayan\")\n        ensure(torch.all(r * (1 - mask) == 0), \"bitişten sonra ödül olmamalı\")\n        ensure(torch.all(r.sum(1) <= 1), \"her bölümde en fazla bir +1\")\n        ensure(logp.requires_grad, \"logp gradyanlı olmalı\")\n    R.test(\"rollout: maske ve ödüller\", ro)\n\n    def tr():\n        torch.manual_seed(0)\n        pi = make_policy()\n        t0 = time.perf_counter()\n        need(ns, \"train_batched\")(pi, Chain(B=64))\n        dt = time.perf_counter() - t0\n        env = Chain(); obs = env.reset(); steps = 0\n        with torch.no_grad():\n            while True:\n                obs, r, done = env.step(pi(obs).argmax(-1)); steps += 1\n                if done.item():\n                    break\n        ensure(env.s.item() == 7 and steps == 4, f\"açgözlü politika {steps} adımda {env.s.item()}. hücrede bitti\")\n        return f\"60 iterasyon × 64 bölüm = 3840 bölüm, {fmt_time(dt)}\"\n    R.test(\"batch'li REINFORCE zinciri çözüyor\", tr)\n    if \"rollout\" not in ns:\n        return\n    torch.manual_seed(0)\n    pi = make_policy()\n    env1, env64 = Chain(B=1), Chain(B=256)\n\n    def seq():\n        for _ in range(256):\n            ns[\"rollout\"](pi, env1, 30)\n    t_seq = bench(seq, repeat=1, number=1)\n    t_bat = bench(lambda: ns[\"rollout\"](pi, env64, 30), repeat=3)\n    R.table(\"256 bölüm × 30 adım\", [\"yöntem\", \"süre\"], [[\"ortamlar tek tek\", fmt_time(t_seq)], [\"256 ortam paralel\", fmt_time(t_bat)]],\n            note=\"Gerçek RL kütüphaneleri (ör. vektörize gym ortamları, GPU simülatörleri) bu yüzden ortamları batch'ler.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m5_5_rl as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**MDP:** durumlar $S$, eylemler $A$, geçiş $P(s'\mid s,a)$, ödül $R(s,a)$, indirim $\gamma\in[0,1)$. Bir politikanın değeri $V^\pi(s)=\mathbb E_\pi\big[\sum_t\gamma^tr_t\mid s_0=s\big]$.

**Bellman denklemleri:**

$$
V^\pi=R_\pi+\gamma P_\pi V^\pi\ \Rightarrow\ V^\pi=(I-\gamma P_\pi)^{-1}R_\pi,\qquad
V^*(s)=\max_a\Big[R(s,a)+\gamma\sum_{s'}P(s'\mid s,a)V^*(s')\Big].
$$

**Value iteration:** ikinci denklemi sabit nokta iterasyonu olarak uygula.

Model bilinmiyorsa ortamla etkileşerek öğreniriz. **policy gradient** (REINFORCE):

$$
\nabla_\theta J=\mathbb E_{\tau\sim\pi_\theta}\Big[\sum_t\nabla_\theta\log\pi_\theta(a_t\mid s_t)\,(G_t-b)\Big],\qquad G_t=\sum_{k\ge t}\gamma^{k-t}r_k .
$$

Kodda bu, `loss = -(log_prob * (G - b)).sum()` kaybının gradyanı.

## 1 · Elle

İki iterasyonda $V(s_1)$: $2\to3$; $V^*(s_1)=4$'e geometrik yaklaşıyor ($4-V_k=4\cdot0.5^k$). $s_0$'da en iyi eylem "git": $1+0.5\cdot4=3$, oysa "kal" $0+0.5\cdot3=1.5$.

Bandit'te $\nabla\log\pi(0)=e_0-\pi=(0.5,-0.5)$. Baseline tek bir örnekte tahmini değiştirir ($0.5\to0.2$), ama beklentiyi değiştirmez: eylem 1 örneklenseydi (ödül 0) $\theta_0$ bileşeni $(0-0.6)\cdot(-0.5)=0.3$ olurdu. Beklenti baseline'lı: $\tfrac12\cdot0.2+\tfrac12\cdot0.3=0.25$; baseline'sız: $\tfrac12\cdot0.5+\tfrac12\cdot0=0.25$. Aynı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İki durum, deterministik. $s_0$: kal → ödül 0, $s_0$'da kal; git → ödül 1, $s_1$'e. $s_1$: kal → ödül 2, $s_1$'de kal; git → ödül 0, $s_0$'a. $\gamma=0.5$, $V_0=0$**

- `v1s0`: $V_1(s_0)$
- `v2s0`: $V_2(s_0)$
- `v2s1`: $V_2(s_1)$
- `vs1`: $V^*(s_1)$
- `vs0`: $V^*(s_0)$
- `qst`: $Q^*(s_0,\text{kal})$

**REINFORCE, iki kollu bandit: $\pi=\operatorname{softmax}(\theta)$, $\theta=(0,0)$; örneklenen eylem 0, ödül 1**

- `g0`: $R\,\partial\log\pi(0)/\partial\theta_0$
- `gb`: Baseline $b=0.6$ ile aynı bileşen: $(R-b)\,\partial\log\pi(0)/\partial\theta_0$

In [ ]:
cevaplar = dict(
    v1s0=None,
    v2s0=None,
    v2s1=None,
    vs1=None,
    vs0=None,
    qst=None,
    g0=None,
    gb=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"v1s0":[1,0],"v2s0":[2,0],"v2s1":[3,0],"vs1":[4,0],"vs0":[3,0],"qst":[1.5,0],"g0":[0.5,0],"gb":[0.2,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. Ortam gym tarzı: `P[s][a]` = `(olasılık, sonraki_durum, ödül, bitti)` listesi; `bitti` ise sonraki durumun değeri 0 sayılır.

- `q_values(P, V, s, nA, gamma)` → `[Q(s, a) for a]`.
- `value_iteration(P, nS, nA, gamma, tol=1e-10)` → `(V, policy)`; $\max_s|V_{k+1}-V_k|<$ `tol` olunca dur. `policy[s]` = argmax (eşitlikte küçük indeks).
- `policy_evaluation(P, policy, nS, gamma, tol=1e-10)` → deterministik `policy` için $V^\pi$ (iteratif).

Kontrol 4×4 kaygan bir buzlu göl (FrozenLake benzeri) üzerinde çalışır.

In [ ]:
%%writefile stage_b.py
def q_values(P, V, s, nA, gamma):
    ...


def value_iteration(P, nS, nA, gamma, tol=1e-10):
    ...


def policy_evaluation(P, policy, nS, gamma, tol=1e-10):
    ...


P = {0: {0: [(1.0, 0, 0.0, False)], 1: [(1.0, 1, 1.0, False)]},
     1: {0: [(1.0, 1, 2.0, False)], 1: [(1.0, 0, 0.0, False)]}}
print(value_iteration(P, 2, 2, 0.5))     # ([3, 4], [1, 0])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; `P: (S, A, S)`, `R: (S, A)` (beklenen ödül).

- `bellman_q(V, P, R, gamma)` → `Q: (S, A)`; `einsum("sat,t->sa", …)`. Döngü yok.
- `policy_eval_exact(P, R, pi, gamma)` — `pi: (S, A)` stokastik politika; $P_\pi$, $R_\pi$ kur, `np.linalg.solve`. Döngü yok.
- `policy_iteration(P, R, gamma)` → `(V, pi_det)`; değerlendir → açgözlü iyileştir → politika değişmeyince dur. Tek döngü.

In [ ]:
%%writefile stage_c.py
import numpy as np


def bellman_q(V, P, R, gamma):
    ...


def policy_eval_exact(P, R, pi, gamma):
    ...


def policy_iteration(P, R, gamma):
    ...


P = np.zeros((2, 2, 2)); P[0, 0, 0] = P[0, 1, 1] = P[1, 0, 1] = P[1, 1, 0] = 1
R = np.array([[0.0, 1.0], [2.0, 0.0]])
print(policy_iteration(P, R, 0.5))   # V = [3, 4], politika [1, 0]

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Ortam (hazır): 8 hücreli zincir, başlangıç 3. hücre, eylemler sol/sağ. Sağ uç +1 ödül ve bitiş, sol uç 0 ödül ve bitiş, en fazla 30 adım. `env.obs()` one-hot durum.

- `Policy(L=8, hidden=32)` — `Linear → Tanh → Linear`, iki logit.
- `discounted_returns(rewards, gamma)` — liste → tensör $G_t$.
- `run_episode(policy, env)` → `(log_probs (T,), rewards listesi)`; `torch.distributions.Categorical(logits=…)`.
- `train_reinforce(policy, env, episodes=300, gamma=0.95, lr=0.01)` — Adam; baseline: başlangıç getirisinin kayan ortalaması `b ← 0.9 b + 0.1 G_0` (ilk bölümde `b = G_0`), kayıp $-\sum_t\log\pi(a_t\mid s_t)(G_t-b)$.

Kontrol eğitilmiş politikanın açgözlü hâliyle 4 adımda sağ uca gittiğini sınar.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


class Chain:
    """B paralel zincir ortamı. Sağ uç +1 ve bitiş, sol uç 0 ve bitiş."""
    def __init__(self, B=1, L=8, max_steps=30, start=3):
        self.B, self.L, self.T, self.start = B, L, max_steps, start

    def reset(self):
        self.s = torch.full((self.B,), self.start)
        self.t = 0
        self.alive = torch.ones(self.B, dtype=torch.bool)
        return self.obs()

    def obs(self):
        return F.one_hot(self.s, self.L).float()

    def step(self, a):
        self.s = torch.where(self.alive, (self.s + 2 * a - 1).clamp(0, self.L - 1), self.s)
        self.t += 1
        r = (self.s == self.L - 1).float() * self.alive
        done = (self.s == 0) | (self.s == self.L - 1) | (self.t >= self.T)
        self.alive = self.alive & ~done
        return self.obs(), r, done


# ---- senin kodun ----

class Policy(nn.Module):
    def __init__(self, L=8, hidden=32):
        super().__init__()
        ...

    def forward(self, obs):
        ...


def discounted_returns(rewards, gamma):
    ...


def run_episode(policy, env):
    ...


def train_reinforce(policy, env, episodes=300, gamma=0.95, lr=0.01):
    ...


torch.manual_seed(0)
pi = Policy()
train_reinforce(pi, Chain())
print(pi(Chain().reset()).softmax(-1))   # sağa gitme olasılığı yüksek olmalı

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Aynı zincir, `B` ortam paralel (`env.alive` maskesi bitmiş ortamları dondurur, ödüllerini sıfırlar).

- `discounted_returns_vec(r, gamma)` — `r: (B, T)` → `G: (B, T)`. **Döngü yok:** $\Gamma_{jt}=\gamma^{j-t}[j\ge t]$ üst üçgen matrisiyle `G = r @ Γ`. Bitiş sonrası ödüller 0 olduğu için doğru.
- `rollout(policy, env, T)` → `(logp (B, T), r (B, T), mask (B, T))`; `mask[:, t]` adım atılmadan önceki `env.alive`. Yalnız zaman üzerinde tek döngü.
- `train_batched(policy, env, iters=60, gamma=0.95, lr=0.01)` — her iterasyonda bir rollout; baseline: her zaman adımında geçerli girdilerin $G$ ortalaması; kayıp $-\frac1B\sum(\text{logp}\cdot(G-b)\cdot\text{mask})$.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F
from torch import nn


class Chain:
    """B paralel zincir ortamı. Sağ uç +1 ve bitiş, sol uç 0 ve bitiş."""
    def __init__(self, B=1, L=8, max_steps=30, start=3):
        self.B, self.L, self.T, self.start = B, L, max_steps, start

    def reset(self):
        self.s = torch.full((self.B,), self.start)
        self.t = 0
        self.alive = torch.ones(self.B, dtype=torch.bool)
        return self.obs()

    def obs(self):
        return F.one_hot(self.s, self.L).float()

    def step(self, a):
        self.s = torch.where(self.alive, (self.s + 2 * a - 1).clamp(0, self.L - 1), self.s)
        self.t += 1
        r = (self.s == self.L - 1).float() * self.alive
        done = (self.s == 0) | (self.s == self.L - 1) | (self.t >= self.T)
        self.alive = self.alive & ~done
        return self.obs(), r, done


def make_policy(L=8, hidden=32):
    return nn.Sequential(nn.Linear(L, hidden), nn.Tanh(), nn.Linear(hidden, 2))


# ---- senin kodun ----

def discounted_returns_vec(r, gamma):
    """(B, T) → (B, T). Döngü yok."""
    ...


def rollout(policy, env, T):
    """→ (logp, r, mask), hepsi (B, T). Tek döngü: zaman."""
    ...


def train_batched(policy, env, iters=60, gamma=0.95, lr=0.01):
    ...


torch.manual_seed(0)
pi = make_policy()
train_batched(pi, Chain(B=64))
print(pi(Chain().reset()).softmax(-1))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Williams (1992): REINFORCE

[Williams — Simple Statistical Gradient-Following Algorithms for Connectionist Reinforcement Learning (Machine Learning 1992)](https://link.springer.com/article/10.1007/BF00992696)

d aşamasındaki REINFORCE'u CartPole'a (gymnasium ya da kendi fizik simülasyonun) taşı; makaledeki "takviye taban çizgisi" (reinforcement baseline) seçeneklerini karşılaştır: taban yok, getirilerin hareketli ortalaması, öğrenilen durum değeri V(s) (M11.9'a köprü). Gradyan tahmincisinin varyansını aynı politikada 100 bölümden tahmin edip tabana göre tablola; taban çizgisinin yansızlığı koruduğunu (beklenen gradyan değişmez) deneyle göster.


**Adımlar**

1. Getirileri bölüm içinde ileriye dönük (reward-to-go) hesaplamanın varyansa etkisini de ölç.

**Kabul ölçütleri**

- Taban çizgisi varyansı belirgin azaltır, ortalama gradyanı değiştirmez.
- CartPole 500 adıma ulaşır.

## Makale ödevi: Sutton, McAllester, Singh, Mansour (2000): politika gradyanı teoremi

[Sutton, McAllester, Singh, Mansour — Policy Gradient Methods for Reinforcement Learning with Function Approximation (NeurIPS 2000)](https://papers.nips.cc/paper/1999/hash/464d828b85b0bed98e80ade0a5c43b0f-Abstract.html)

c aşamasındaki tablo MDP'de (geçişler ve ödüller bilinirken) politika gradyanı teoremini sayısal olarak doğrula: $\nabla J=\sum_sd^\pi(s)\sum_a\nabla\pi(a|s)Q^\pi(s,a)$'yı kapalı formda (durum ziyaret dağılımını doğrusal sistemle çözerek) hesapla, sonlu farklarla karşılaştır. Teorem 2'nin "uyumlu fonksiyon yaklaşımı" koşulunu sağlayan doğrusal bir Q yaklaşımıyla gradyanın hâlâ yansız olduğunu göster.


**Adımlar**

1. Softmax politika, durum başına tablo parametreleri.

**Kabul ölçütleri**

- Kapalı form gradyan = sonlu farklar (10⁻⁶ içinde).
- Uyumlu yaklaşım yansız, uyumsuz bir yaklaşım yanlı.

## Kaynaklar

- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı)](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Williams — Simple Statistical Gradient-Following Algorithms (REINFORCE)](https://link.springer.com/article/10.1007/BF00992696) — ©
- [OpenAI Spinning Up — Intro to Policy Optimization](https://spinningup.openai.com/) — MIT
- [UC Berkeley CS285 — Deep Reinforcement Learning](https://rail.eecs.berkeley.edu/deeprlcourse/) — doğrulanmadı
- [David Silver — UCL RL Course](https://www.davidsilver.uk/teaching/) — doğrulanmadı